In [9]:
import pandas as pd

### 데이터 프레임 병합

In [6]:
file_paths = {
    "HYBE": "./data_0505/HYBE논란_라벨링01.csv",
    "JYP": "./data_0505/JYP논란_라벨링01.csv",
    "SM": "./data_0505/SM논란_라벨링01.csv",
    "YG": "./data_0505/YG논란_라벨링01.csv"
}

# 각 데이터프레임 로딩 및 소속사 컬럼 추가
df_list = []
for agency, path in file_paths.items():
    df = pd.read_csv(path)
    df_list.append(df)

# 병합
merged_df = pd.concat(df_list, ignore_index=True)
merged_df.to_csv('./아이돌논란_라벨링01.csv', index = False)

### 1. CSV 로드 및 전처리

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
from tqdm import tqdm
import re

# print 출력옵션
pd.set_option('display.max_colwidth', None)

In [34]:
df =pd.read_csv('./아이돌논란_라벨링02.csv')
df.head(3)

,소속사,그룹,연예인 이름,사건 날짜,사건 내용,라벨 1,라벨 2,라벨 3
0,HYBE,방탄소년단,슈가,2020년 5월 17일,"2020년 5월 17일 발매된 슈가의 믹스테잎 D-2 의 수록곡 <어떻게 생각해?>의 도입부에 인민사원 집단 자살 사건 의 주범으로 악명높은 짐 존스 의 육성 연설이 샘플링되어 삽입되어 있다는 게 밝혀져 논란이 되었다. 이에 대해 슈가가 악플러들에 대한 반어적 비유를 사용하기 위해 짐 존스의 연설을 삽입한 거 아니냐는 의견과 존스타운과 전혀 관계없는 노래에 굳이 범죄자의 육성을 삽입한 이유가 궁금하다는 의견 등등 갑론을박이 벌어졌다. 인민사원 집단 자살 사건 이 국내에서는 별로 알려진 사건이 아니지만, 해당 사건이 일어난 국가 내에서는 꾸준히 교육시킬 정도로 심각했던 사건인 것은 사실이다. 곡에 담긴 연설의 부분은 ""당신은 죽더라도 살 것이다. 살아서 믿는 자는 결코 죽지 않을 것이다.""라는 내용이다. 다만 원래 이 문장은 요한복음 11장 25절에 있는 구절이다. 즉, 사이비 목사가 성경 내용을 악의적으로 이용한 것이다. 하지만 해당 연설 내용은 집단 자살 사건을 의미하기 때문에 문제가 된다. 해외에서도 짐 존스를 소재로 삼은 곡들이 꽤 있다. 포스트 말론의 '존스타운'과 릴 우지 버트의 'Leaders'는 가사에서 짐 존스가 언급되며, BROCKHAMPTON 의 '1998 TRUMAN'는 짐 존스의 음성이 들어간다. 그러나 이 곡들은 모두 맥락의 부합성 면에서 슈가의 노래와 큰 차이점을 보인다. '존스타운'의 경우는 2분이 채 되지 않는 짧은 곡으로 앞 트랙의 흐름을 이어가는 역할이고, 'Leaders'의 경우에는 '짐 존스처럼 교단을 이끌어'라며 자신의 인기를 과시하기 위한 비유로 써 먹었다. '1998 TRUMAN'은 거대한 환상같은 현실 속 방탕한 삶을 사는 청년들의 현실을 폭로하는 내용으로, '너흰 자유가 아냐, 너흰 노예야!'라 외치는 짐 존스의 육성이 곡의 테마를 잡는다. 해당 아티스트들이 곡에서 짐 존스를 언급한 의도는 각자 다르고, 이에 대한 평가도 '범죄자 미화다', '예술의 영역이다', '수많은 피해자를 낳은 범죄자인데 육성 삽입 자체가 적절치 못했다' 등등 다 달랐다. 하여튼 슈가의 노래와 달리 짐 존스가 왜 등장했는지 그 의도와 맥락의 부합성이 분명하다는 공통점이 있다. 해명 논란이 일자 빅히트 엔터테인먼트(현 빅히트 뮤직 )는 ""도입부 연설 보컬 샘플은 해당 곡의 트랙을 작업한 프로듀서가 특별한 의도 없이 연설자를 알지 못한 상태에서 곡 전체의 분위기를 고려해 선정했다. 해당 연설 보컬 샘플을 선정한 이후 회사는 내부 프로세스에 따라 내용의 적정성을 확인하는 절차를 진행했다. 하지만 선정 및 검수 과정에서 내용상 부적절한 샘플임을 인지하지 못하고 곡에 포함하는 오류가 있었다.""며 해명하고 공식 사과했다. # 이후 연설 음성을 삭제한 음원을 재발매하였다. 이후 같은 곡에서 베트남 전범 범죄자의 연설을 사용했다는 루머가 추가로 나돌기도 하였으나 소속사인 빅히트 엔터테인먼트 측에서 ""사실무근""이라며 이를 일축했다. 해당 소문은 단순 허위사실 루머인 것으로 확인되었다. # 확인 결과, 논란이 된 짐 존스의 연설 샘플은 빅히트 프로듀서가 직접 짐 존스 연설 원본에서 샘플링한 것이 아니라 스플라이스라는 샘플 플랫폼에서 판매되고 있는 샘플이었다. # 해당 곡의 트랙을 작업한 빅히트 프로듀서는 연설자가 누군지 모르고 실수로 해당 연설 보컬 샘플을 선정한 것으로 추측된다. 또한 짐 존스 연설 샘플이 사용된 트랙 <어떻게 생각해?>의 크레딧을 확인해보면 슈가가 프로듀서 명단에 들어가있는 다른 곡들과 다르게 해당 곡은 EL CAPITXN , GHSTLOOP만 프로듀서 명단에 들어가 있는 것을 확인할 수 있다. 즉, 실수로 해당 연설 보컬 샘플을 선정한 프로듀서는 슈가가 아니며 슈가는 이 노래에는 트랙이 아니라 작사, 녹음 등의 작업에만 참여했다.",발언 문제,사회적 감수성,종교/이념
1,HYBE,방탄소년단,슈가,2020년 5월 29일,"2020년 5월 29일, 진행한 V LIVE 에서 슈가는 팬들 에게 총 10곡이 담긴 솔로 믹스테이프 ‘ D-2 ’의 작업 뒷이야기를 전하며 ‘대취타’와 ‘인터루드 : 셋 미 프리’(Interlude : Set Me Free) 작업을 끝내고 믹스테이프에 수록할 수 있었던 것에 대해 “코로나19가 가져다준 행운”이라고 말했다. 이어 “코로나19 ‘때문’이 아닌 코로나19 ‘덕분’”이라고 재차 강조했다. 이후 온라인상에서는 슈가가 전 세계를 패닉에 빠트린 코로나19에 ‘행운’이라는 단어와 긍정적 맥락에 쓰이는 ‘덕분’라는 표현을 가져다 붙인 것을 두고 부적절했다는 반응이 이어졌다. #",발언 문제,사회적 감수성,NaN
2,HYBE,방탄소년단,RM,2016년\r\n7월 6일,"2016년 7월 6일 방탄소년단의 소속사 빅히트 엔터테인먼트 는 여성혐오 가사 논란에 대해 죄송하고 피드백을 받겠다는 공식 입장을 밝혔다. # 입장 발표 이후 해당 논란을 주로 제기하던 트위터 계정 도 '피드백에 기쁨을 느끼며 앞으로도 응원하겠다'는 입장을 밝히면서 사건은 일단락되었다. 한편, 출판사 열린책들 에서는 방탄소년단의 가사를 키워드로 한 기획 기사를 내보내면서 이 문제를 지적하기도 했다. # 다만, 방탄소년단 측에서 이러한 지적을 수긍했다는 점을 언급하며, 최근 앨범 수록곡인 <21세기 소녀>에 대해서는 소녀팬들에게 자기 긍정의 메시지를 전달했다는 점을 높이 평가했다. 다만 2015~2016년 방탄소년단의 여성혐오 가사 논란 당시 관련 인터넷 커뮤니티에서 상당히 많이 제기된 의혹이 있었는데, 가사에 문제를 제기했던 사람들이 사실 타 그룹 팬덤이고, 적극적으로 이슈화에 앞장선 건 남성 혐오 성향이 있는 사람들이라는 것이다. 즉, 순수한 목적으로 문제를 제기한 게 아니었다는 것인데, 어쨌든 당시 방탄소년단은 무명에 가까웠던 시절을 끝내고 드디어 국내에서 인지도를 올리고 팬덤을 키워가던 시점이었기 때문에 사회 분위기를 고려하여 빠르게 문제를 해결하는 것을 선택했다. 문제가 된 가사들은 개사하여 부르게 되었다. 여성혐오 가사 논란 이후 RM 은 페미니즘 관련 도서를 읽는 모습을 보여주었다. 해외 언론에서도 방탄소년단을 주목해야 할 이유로 정신적 고뇌(Whalien 52), 정치(Change), 여성 인권(21세기 소녀)을 노래한다는 점을 언급했다. 링크 한편, 피드백 이후 2017년 고척 스카이돔 콘서트 메들리 때 과거 히트곡 <호르몬 전쟁>의 일부 소절을 부른 것에 대해 일부에서는 이제 <호르몬 전쟁>은 아예 부르지도 말아야 된다고 주장 했는데, 해당기사 댓글 반응을 보면 그러한 지적에 동의하지 않는 반응이 있었다. 참고로 해당 곡 발표 당시에 어느 여성 에디터의 반응은 30대 여성이 듣기엔 오글거림이 심하다는 것이었는데, 여성혐오 라 해석하지는 않았다. 링크 어쨌든 해당 문제 제기도 받아들여 이 이후부터는 부르지 않았다.",사회적 감수성,발언 문제,NaN


In [35]:
# 전처리: 사건 내용 정제 + 불용어 제거

# 조사
particles_to_remove = {
    "은", "는", "이", "가", "을", "를", "에", "의", "도", "과", "와",
    "에서", "부터", "까지", "만", "든", "로", "으로", "라도", "조차"
}

# 한 글자지만 의미가 중요한 단어
important_single_chars = {"팬", "군", "죄", "법", "술", "약"}

with open('./korean_stopwords.txt', encoding='utf-8') as f:
    stopwords = set(w.strip() for w in f.readlines() if w.strip())

def clean_text(text):
    if pd.isnull(text):
        return ""
    
     # 1. 특수문자 및 한글/숫자/공백 이외 제거
    text = re.sub(r"[^가-힣0-9\s]", " ", text)
    
    # 2. 중복 공백 제거 및 좌우 공백 제거
    text = re.sub(r"\s+", " ", text).strip()
    
    # 3. 불용어 목록에 있는 단어 제거
    tokens = []
    for t in text.split():
        if t in stopwords:
            continue
        if t in particles_to_remove:
            continue
        if len(t) == 1 and t not in important_single_chars:
            continue
        tokens.append(t)
    
    return " ".join(tokens)

df['clean_text'] = df['사건 내용'].apply(clean_text)

In [36]:
print(df.clean_text.head(1))

0    2020년 5월 17일 발매된 슈가의 믹스테잎 수록곡 생각해 도입부에 인민사원 집단 자살 사건 주범으로 악명높은 존스 육성 연설이 샘플링되어 삽입되어 있다는 밝혀져 논란이 되었다 이에 대해 슈가가 악플러들에 대한 반어적 비유를 사용하기 위해 존스의 연설을 삽입한 아니냐는 의견과 존스타운과 전혀 관계없는 노래에 굳이 범죄자의 육성을 삽입한 이유가 궁금하다는 의견 갑론을박이 벌어졌다 인민사원 집단 자살 사건 국내에서는 별로 알려진 사건이 아니지만 해당 사건이 일어난 국가 내에서는 꾸준히 교육시킬 정도로 심각했던 사건인 것은 사실이다 곡에 담긴 연설의 부분은 당신은 죽더라도 것이다 살아서 믿는 자는 결코 죽지 않을 것이다 라는 내용이다 원래 문장은 요한복음 11장 25절에 있는 구절이다 사이비 목사가 성경 내용을 악의적으로 이용한 것이다 해당 연설 내용은 집단 자살 사건을 의미하기 문제가 된다 해외에서도 존스를 소재로 삼은 곡들이 포스트 말론의 존스타운 우지 버트의 가사에서 존스가 언급되며 1998 존스의 음성이 들어간다 곡들은 맥락의 부합성 면에서 슈가의 노래와 차이점을 보인다 존스타운 경우는 2분이 되지 않는 짧은 곡으로 트랙의 흐름을 이어가는 역할이고 경우에는 존스처럼 교단을 이끌어 라며 자신의 인기를 과시하기 위한 비유로 먹었다 1998 거대한 환상같은 현실 방탕한 삶을 사는 청년들의 현실을 폭로하는 내용으로 너흰 자유가 아냐 너흰 노예야 외치는 존스의 육성이 곡의 테마를 잡는다 해당 아티스트들이 곡에서 존스를 언급한 의도는 다르고 이에 대한 평가도 범죄자 미화다 예술의 영역이다 수많은 피해자를 낳은 범죄자인데 육성 삽입 자체가 적절치 못했다 달랐다 하여튼 슈가의 노래와 달리 존스가 등장했는지 의도와 맥락의 부합성이 분명하다는 공통점이 해명 논란이 일자 빅히트 엔터테인먼트 빅히트 뮤직 도입부 연설 보컬 샘플은 해당 곡의 트랙을 작업한 프로듀서가 특별한 의도 없이 연설자를 알지 못한 상태에서 전체의 분위기를 고려해 선정했다 해당 연설 보컬 샘플을 선정

### 2. 라벨 처리 (다중 라벨)

In [37]:
df['label_list'] = df[['라벨 1', '라벨 2', '라벨 3']].values.tolist()
df['label_list'] = df['label_list'].apply(lambda x: [i for i in x if pd.notnull(i)])

all_labels = [
    "병역 문제", "범죄 혐의", "세금 문제", "성 관련", "사생활", "팬 대응",
    "발언 문제", "사회적 감수성", "종교/이념", "혐의정보 유포", "무의식적 태도", "기타"
]
mlb = MultiLabelBinarizer(classes=all_labels)
y = mlb.fit_transform(df['label_list'])

In [38]:
print(mlb.classes_) # 핫인코딩 순서
print(y) #  핫인코딩한 결과물

['병역 문제' '범죄 혐의' '세금 문제' '성 관련' '사생활' '팬 대응' '발언 문제' '사회적 감수성' '종교/이념'
 '혐의정보 유포' '무의식적 태도' '기타']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 1]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 1 0 0]]


### 3. Tokenizer 및 Dataset 구성

In [39]:
from transformers import AutoTokenizer

# KoBERT 사전학습 모델 호출
tokenizer = AutoTokenizer.from_pretrained("skt/kobert-base-v1")

# 입력 텍스트 목록 생성
texts = df['clean_text'].tolist()

class NewsDataset(Dataset):
    
    # 초기화 함수
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts              # 입력 문장 리스트
        self.labels = labels            # 라벨 벡터 리스트 (다중 라벨: [0, 1, 0, ..., 1])
        self.tokenizer = tokenizer      # HuggingFace tokenizer (KoBERT용)
        self.max_len = max_len          # 입력 시퀀스 최대 길이

    # 데이터셋의 총 샘플 수 반환
    def __len__(self):
        return len(self.texts)

    # 하나의 샘플을 인덱스로 불러올 때 실행   
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.FloatTensor(self.labels[idx])
        }


### 4. 데이터 분할 및 DataLoader

In [40]:
# 전체 데이터셋 → train:val:test = 72:18:10 비율로 분할
X_temp, X_test, y_temp, y_test = train_test_split(texts, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

train_dataset = NewsDataset(X_train, y_train, tokenizer)
val_dataset = NewsDataset(X_val, y_val, tokenizer)
test_dataset = NewsDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)


### 5. KoBERT 모델 정의

In [41]:
class KoBERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels=12):
        super().__init__()
        self.bert = BertModel.from_pretrained("skt/kobert-base-v1")                 # KoBERT 사전학습 모델 로드
        self.classifier = nn.Linear(768, num_labels)                                # 분류기 정의: KoBERT 출력(768차원) → 라벨 수만큼의 출력 (12개)

    # 순전파(forward) 정의
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)     # KoBERT 통과: 입력 문장을 BERT 구조로 인코딩해서 모든 토큰의 문맥 벡터를 계산
        pooled_output = outputs.pooler_output                                       # pooled_output: [CLS] 토큰에 해당하는 문장 전체 표현 (batch_size x 768)
        logits = self.classifier(pooled_output)                                     # 분류기 통과 → 라벨별 예측 로짓 출력 (batch_size x num_labels) → 추후 활성화함수를 통해 확률로 변환예정
        return logits

### 6. 학습 설정

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KoBERTMultiLabelClassifier().to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# pos_weight 계산 (클래스 불균형 보정용)
class_counts = y.sum(axis=0)
total_samples = len(y)
pos_weights = torch.tensor((total_samples - class_counts) / class_counts, dtype=torch.float32).to(device)

# 불균형 보정된 Loss 함수
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = KoBERTMultiLabelClassifier().to(device)

# optimizer = AdamW(model.parameters(), lr=2e-5)
# criterion = nn.BCEWithLogitsLoss()

### 7. 학습 루프

In [43]:
for epoch in range(3):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}")

    # 검증
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].cpu().numpy()
            logits = model(input_ids, attention_mask)
            probas = torch.sigmoid(logits).cpu().numpy()
            preds.append(probas)
            trues.append(labels)

    preds = np.vstack(preds)
    trues = np.vstack(trues)
    pred_labels = (preds > 0.1).astype(int)
    f1 = f1_score(trues, pred_labels, average='macro')
    print(f"Validation F1 Score: {f1:.4f}")

100%|██████████| 12/12 [00:02<00:00,  4.92it/s]


Epoch 1 - Train Loss: 1.2693
Validation F1 Score: 0.1919


100%|██████████| 12/12 [00:02<00:00,  5.54it/s]


Epoch 2 - Train Loss: 1.2275
Validation F1 Score: 0.1919


100%|██████████| 12/12 [00:02<00:00,  5.53it/s]


Epoch 3 - Train Loss: 1.2337
Validation F1 Score: 0.1919


In [44]:
from sklearn.metrics import classification_report

model.eval()
preds, trues = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        logits = model(input_ids, attention_mask)
        probas = torch.sigmoid(logits).cpu().numpy()

        preds.append(probas)
        trues.append(labels)

preds = np.vstack(preds)
trues = np.vstack(trues)
pred_labels = (preds > 0.5).astype(int)

print(f"Test F1 (macro): {f1_score(trues, pred_labels, average='macro'):.4f}")
print(classification_report(trues, pred_labels, target_names=mlb.classes_))


Test F1 (macro): 0.0889
              precision    recall  f1-score   support

       병역 문제       0.00      0.00      0.00         3
       범죄 혐의       0.17      0.50      0.25         2
       세금 문제       0.05      1.00      0.09         1
        성 관련       0.00      0.00      0.00         3
         사생활       0.00      0.00      0.00         4
        팬 대응       0.00      0.00      0.00         2
       발언 문제       0.00      0.00      0.00         0
     사회적 감수성       0.00      0.00      0.00         0
       종교/이념       0.00      0.00      0.00         0
     혐의정보 유포       0.00      0.00      0.00         3
     무의식적 태도       0.10      1.00      0.17         2
          기타       0.38      1.00      0.55         8

   micro avg       0.09      0.43      0.15        28
   macro avg       0.06      0.29      0.09        28
weighted avg       0.13      0.43      0.19        28
 samples avg       0.10      0.29      0.14        28



c:\Users\wjdrl\anaconda3\envs\jsy\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\wjdrl\anaconda3\envs\jsy\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\wjdrl\anaconda3\envs\jsy\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [48]:
# 원 데이터 라벨 분포
from collections import Counter

# 다중 리스트 안의 개별 라벨들 모두 펼쳐서 세기
flat_labels = [label for sublist in df['label_list'] for label in sublist]
label_counter = Counter(flat_labels)

# 보기 좋게 출력
for label, count in label_counter.items():
    print(f"{label}: {count}건")

발언 문제: 33건
사회적 감수성: 14건
종교/이념: 7건
세금 문제: 4건
팬 대응: 36건
기타: 59건
사생활: 46건
혐의정보 유포: 37건
범죄 혐의: 43건
무의식적 태도: 47건
병역 문제: 27건
성 관련: 13건


=> 증강 or 데이터셋 추가 ㅋㅋㅋ